In [93]:
import pandas as pd

df_train = pd.read_csv("train_data.csv")
df_test = pd.read_csv("test_data.csv")

df_train.head()

,User_ID,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Gender,Calories
0,14498311,79,165,73,13,79,40.0,male,54
1,16700865,28,148,48,6,91,39.4,female,28
2,17197650,27,187,82,25,101,40.8,male,128
3,16400540,40,173,71,16,93,40.4,male,69
4,15539631,40,183,90,5,80,39.0,male,14


In [94]:
subtask1 = len(df_train)
subtask2 = len(df_train[df_train.Gender == 'male'])
subtask3 = df_train.Duration.mean()
subtask4 = len(df_train[df_train.Age >= 75])

In [95]:
subtasks = pd.DataFrame({
    "subtaskID": [1, 2, 3, 4],
    "datapointID": [1, 1, 1, 1],
    "answer": [subtask1, subtask2, subtask3, subtask4]
})

subtasks

,subtaskID,datapointID,answer
0,1,1,9000.000000
1,2,1,4443.000000
2,3,1,15.510667
3,4,1,412.000000


In [96]:
#preproceessing

df_train.Gender = df_train.Gender.map({"female": 0, "male": 1})
df_test.Gender = df_test.Gender.map({"female": 0, "male": 1})

df_test.Gender = df_test.Gender.fillna(df_test.Gender.median())

p5 = df_test[df_test['Subtask'] == 5]
p6 = df_test[df_test['Subtask'] == 6]

features = ['Age', 'Height', 'Weight', 'Duration', 'Heart_Rate', 'Body_Temp', 'Gender']
target = 'Calories'

X = df_train[features]
y = df_train[target]

X_1 = p5[features]
X_2 = p6[features]

p6.head()

,Subtask,User_ID,Age,Height,Weight,Duration,Heart_Rate,Body_Temp,Gender
2000,6,12214037,23,189,91,24,101,40.6,1.0
2001,6,10678968,35,175,77,12,90,40.1,1.0
2002,6,18945548,29,196,91,22,96,40.6,1.0
2003,6,12725801,31,181,83,4,81,38.6,1.0
2004,6,10722301,25,173,72,22,93,40.5,1.0


In [105]:
from sklearn.model_selection import cross_val_score
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression
from catboost import CatBoostRegressor

model = CatBoostRegressor(random_state=42, verbose=0)

cross_val_score(model, X, y, scoring='neg_mean_absolute_error').mean()

-0.5758273066685439

In [106]:
model.fit(X, y)
y_1 = model.predict(X_1)
y_2 = model.predict(X_2)
y_1

array([113.04785305, 117.54242846,  94.15840026, ..., 188.25925143,
       176.21929071, 101.09974647])

In [107]:
preds_1 = pd.DataFrame({
    "subtaskID": 5,
    "datapointID": p5.User_ID,
    "answer": y_1
})

preds_2 = pd.DataFrame({
    "subtaskID": 6,
    "datapointID": p6.User_ID,
    "answer": y_2
})

preds = pd.concat([preds_1, preds_2])
preds

,subtaskID,datapointID,answer
0,5,12618012,113.047853
1,5,19091177,117.542428
2,5,14793975,94.158400
3,5,17338539,145.498941
4,5,16351902,14.739269
...,...,...,...
2495,6,12838170,29.363027
2496,6,19782827,165.655671
2497,6,19941237,10.830109
2498,6,18089588,33.497655


In [108]:
submission = pd.concat([subtasks, preds])
submission

,subtaskID,datapointID,answer
0,1,1,9000.000000
1,2,1,4443.000000
2,3,1,15.510667
3,4,1,412.000000
0,5,12618012,113.047853
...,...,...,...
2495,6,12838170,29.363027
2496,6,19782827,165.655671
2497,6,19941237,10.830109
2498,6,18089588,33.497655


In [109]:
submission.to_csv("output.csv", index=False)